In [ ]:
import sys
!{sys.executable} -m pip install langchain-openai python-dotenv pdfplumber
!{sys.executable} -m pip install langchain

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import operator
from typing import Annotated, Any
from typing_extensions import TypedDict
from dotenv import load_dotenv

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables.config import RunnableConfig

# Cargar variables de entorno
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Clientes de IA
client = OpenAI(api_key="--") # Para Embeddings
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="---"
)

In [ ]:
def generar_embedding(texto):
    response = client.embeddings.create(
        input=texto,
        model="text-embedding-ada-002"
    )
    return response.data[0].embedding

# Cargar la bitácora histórica
ruta_bitacora = "historico.json"
with open(ruta_bitacora, "r", encoding="utf-8") as f:
    bitacora_raw = json.load(f)

bitacora_procesada = []

print("Limpiando datos y generando embeddings para la bitácora...")
for item in bitacora_raw:
    # Descartamos 'pagina' y 'archivo_origen'. Solo usamos subtitulo y fragmento.
    texto_contexto = f"Incidente: {item['subtitulo']}\nDetalle y Solución Aplicada: {item['fragmento']}"

    bitacora_procesada.append({
        "texto_rag": texto_contexto,
        "embedding": generar_embedding(texto_contexto)
    })

print(f"¡Base RAG lista con {len(bitacora_procesada)} soluciones históricas!")

Limpiando datos y generando embeddings para la bitácora...
¡Base RAG lista con 80 soluciones históricas!


In [ ]:
# Cargar el archivo JSON con las conversaciones
ruta_json = "transcripciones.json"

with open(ruta_json, "r", encoding="utf-8") as f:
    conversaciones = json.load(f)

print(f"Total de conversaciones a procesar en paralelo: {len(conversaciones)}")

Total de conversaciones a procesar en paralelo: 80


In [ ]:
# PROMPTS
systemPrompt = """
Eres un asistente de IA especializado en la extracción de datos de transcripciones de soporte técnico de TI.
Tu tarea es analizar la conversación, identificar datos clave y recomendar una solución basada ESTRICTAMENTE en la bitácora histórica proporcionada.

BITÁCORA HISTÓRICA DE SOLUCIONES (Contexto RAG):
####
{contexto_rag}
####

Conversación a analizar:
####
{texto_conversacion}
####
"""

userPrompt = """
Extrae la siguiente información de la conversación y devuelve un JSON con las siguientes claves exactas:
- "NombreUsuario"
- "Area"
- "IdUsuario"
- "Resumen" (Un resumen breve del problema en 1 o 2 oraciones)
- "Fecha"
- "Persistencia" (Indicar si el problema ya ha pasado antes estrictamente con "Si", "No" o "No hay información)
- "Tipo" (Clasificar estrictamente como "Hardware", "Software" o "No hay información")
- "IdEquipo"
- "Teléfono"
- "Correo"
- "IdTicket"
- "NombreTi"
- "Persona1" (Indicar si es "Usuario" o "Personal de TI")
- "RecomendacionSolucion" (Recomienda una solución basándote ÚNICAMENTE en la 'BITÁCORA HISTÓRICA DE SOLUCIONES' provista. Si el contexto RAG no ayuda a resolver el problema de la conversación, coloca "Escalar a nivel 2").

IMPORTANTE:
- Si no encuentras la información, el valor debe ser estrictamente "No hay información".
- No inventes ni asumas datos que no estén explícitos en el texto.
- Tu respuesta debe ser ÚNICAMENTE un objeto JSON válido, sin bloque de código markdown (sin ```json).
"""

In [ ]:
# ----------------------------------------------------------------------------------------
# ----------------------------- ESTADO DE LOS NODOS  -------------------------------------
# ----------------------------------------------------------------------------------------
class State(TypedDict):
    aggregate: Annotated[list, operator.add]

# ----------------------------------------------------------------------------------------
# ----------------------------- CLASE GENERAL DE NODOS  ----------------------------------
# ----------------------------------------------------------------------------------------
class ReturnNodeValueRAG:
    def __init__(self, node_name: str, llm_model, system_prompt: str, user_prompt: str, texto_conversacion: str, base_conocimiento: list):
        self.node_name = node_name  # Nombre del nodo
        self.llm_model = llm_model   # Conexión al modelo LLM
        self.system_prompt = system_prompt  # Prompt de sistema (ahora incluye espacio para contexto)
        self.user_prompt = user_prompt      # Prompt de usuario
        self.texto_conversacion = texto_conversacion  # La conversación a analizar
        self.base_conocimiento = base_conocimiento    # La bitácora vectorizada

    def __call__(self, state: State) -> Any:
        try:
            # 1. Búsqueda RAG: Generar embedding de la conversación actual
            embedding_conversacion = generar_embedding(self.texto_conversacion)

            # 2. Comparación Semántica (Similitud de cosenos)
            similitudes = []
            vec1 = np.array(embedding_conversacion)

            for dicc in self.base_conocimiento:
                vec2 = np.array(dicc["embedding"])
                similitud = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
                similitudes.append(similitud)

            # 3. Escoger la mejor solución histórica
            best_index = np.argmax(similitudes)
            best_score = similitudes[best_index]

            # Solo inyectamos el contexto si supera el umbral del 78% de similitud
            contexto_rag = "No hay información histórica relevante."
            if best_score > 0.78:
                contexto_rag = self.base_conocimiento[best_index]["texto_rag"]

            # 4. Formatear Prompts e invocar al LLM
            prompt_sistema_final = self.system_prompt.format(
                contexto_rag=contexto_rag,
                texto_conversacion=self.texto_conversacion
            )

            messages = [
                ("system", prompt_sistema_final),
                ("human", self.user_prompt)
            ]

            respuestaIA = self.llm_model.invoke(messages)
            return {"aggregate": [{self.node_name: respuestaIA.content}]}

        except Exception as e:
            print(f"Error al ejecutar el agente para {self.node_name}: {e}")
            # Retornamos un JSON vacío para que el error de un nodo no detenga todo el lote
            return {"aggregate": [{self.node_name: "{}"}]}

In [ ]:
import time
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables.config import RunnableConfig

# Cargar el archivo JSON con las conversaciones a evaluar
# (Asegúrate de que la ruta sea correcta según tu entorno)
with open("transcripciones.json", "r", encoding="utf-8") as f:
    conversaciones = json.load(f)

# Configuración de los lotes para proteger la API (Rate Limits)
tamaño_lote = 16
tiempo_espera = 0.5
resultados_globales = []

print(f"Iniciando procesamiento RAG + Paralelo de {len(conversaciones)} conversaciones en lotes de {tamaño_lote}...")

for i in range(0, len(conversaciones), tamaño_lote):
    lote = conversaciones[i : i + tamaño_lote]
    numero_lote = (i // tamaño_lote) + 1

    print(f"\n--- Procesando Lote {numero_lote} (Conversaciones {i + 1} a {min(i + tamaño_lote, len(conversaciones))}) ---")

    builder = StateGraph(State)

    for index, conversacion in enumerate(lote):
        id_global = i + index
        node_name = f"nodo{id_global}"

        # Convertir a texto la conversación actual
        texto_conv = json.dumps(conversacion, ensure_ascii=False)

        # Instanciar el nodo RAG pasándole la bitácora procesada en la Celda 2
        nodo = ReturnNodeValueRAG(node_name, llm, systemPrompt, userPrompt, texto_conv, bitacora_procesada)

        builder.add_node(node_name, nodo)
        builder.add_edge(START, node_name)
        builder.add_edge(node_name, END)

    # Compilar y ejecutar el grafo del lote actual
    graph = builder.compile()
    config_run = RunnableConfig(recursion_limit=50)

    try:
        llm_response = graph.invoke({"aggregate": []}, config_run)
        resultados_globales.extend(llm_response['aggregate'])
        print(f"Lote {numero_lote} completado con éxito.")
    except Exception as e:
        print(f"Error general en el Lote {numero_lote}: {e}")

    # Pausa de seguridad antes del siguiente lote (excepto si es el último)
    if i + tamaño_lote < len(conversaciones):
        print(f"Esperando {tiempo_espera} segundos para cuidar los límites de la API...")
        time.sleep(tiempo_espera)

print("\n¡Procesamiento de todos los lotes finalizado!")

Iniciando procesamiento RAG + Paralelo de 80 conversaciones en lotes de 16...

--- Procesando Lote 1 (Conversaciones 1 a 16) ---
Lote 1 completado con éxito.
Esperando 0.5 segundos para cuidar los límites de la API...

--- Procesando Lote 2 (Conversaciones 17 a 32) ---
Lote 2 completado con éxito.
Esperando 0.5 segundos para cuidar los límites de la API...

--- Procesando Lote 3 (Conversaciones 33 a 48) ---
Lote 3 completado con éxito.
Esperando 0.5 segundos para cuidar los límites de la API...

--- Procesando Lote 4 (Conversaciones 49 a 64) ---
Lote 4 completado con éxito.
Esperando 0.5 segundos para cuidar los límites de la API...

--- Procesando Lote 5 (Conversaciones 65 a 80) ---
Lote 5 completado con éxito.

¡Procesamiento de todos los lotes finalizado!


In [ ]:
# ----------------------------------------------------------------------------------------
# ------------------ Generación de Resultados y Exportación a Excel ----------------------
# ----------------------------------------------------------------------------------------
datos_extraidos = []

# Extraer y aplanar la respuesta estructurada de nuestra variable global
resultado_diccionario = {list(item.keys())[0]: list(item.values())[0] for item in resultados_globales}

for llave, valor in resultado_diccionario.items():
    raw_response = valor.strip()

    # Limpieza por si el LLM devuelve formato markdown (```json ... ```) a pesar de la advertencia
    if raw_response.startswith("```json"):
        raw_response = raw_response[7:]
    if raw_response.endswith("```"):
        raw_response = raw_response[:-3]

    try:
        if raw_response == "{}":
            continue # Saltamos errores capturados

        datos_json = json.loads(raw_response.strip())
        datos_extraidos.append(datos_json)
    except json.JSONDecodeError as e:
        print(f"Error al parsear el JSON en el {llave}: {e}")

# Crear DataFrame y exportar
if datos_extraidos:
    df_resultados = pd.DataFrame(datos_extraidos)

    # Define el nombre de tu archivo (puedes ponerle "data/Reporte.xlsx" o solo "Reporte.xlsx")
    archivo_salida = "Reporte_Tickets_TI.xlsx"

    # Obtenemos el directorio del archivo
    directorio = os.path.dirname(archivo_salida)

    # Solo intentamos crear la carpeta si hay un directorio especificado
    if directorio:
        os.makedirs(directorio, exist_ok=True)

    # Guardamos en Excel
    df_resultados.to_excel(archivo_salida, index=False)
    print(f"\n¡Éxito! Se procesaron y guardaron {len(datos_extraidos)} registros en: {archivo_salida}")
else:
    print("\nNo se pudieron extraer datos válidos para exportar.")


¡Éxito! Se procesaron y guardaron 80 registros en: Reporte_Tickets_TI.xlsx
